Pydantic AI is not giving me enough information to debug why the LLM is looping when I ask for a penthouse suite (which doesn't exist). So I'm going to try LangChain.

In [9]:
import os

from dotenv import load_dotenv

load_dotenv("../.env")
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace("postgresql", "postgresql+psycopg")

gpt_model = "gpt-5-mini"

Get the enums

In [10]:
import psycopg

enums = ["availability_status_type", "room_bed_type", "room_status_type", "room_type"]
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]


print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

Write out the database schema and setup the connection for the LLM.

In [11]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}, do not return
                last_renovation DATE, -- Do not provide unless asked for
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY, -- Do not return
                room_id INT NOT NULL, -- Do not return
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)

Setup the LangChain database toolkit

In [13]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_openai import ChatOpenAI
from pydantic_ai.ext.langchain import LangChainToolset

# 1. Initialize your Language Model (LLM)
# Ensure your LLM supports Pydantic AI/Function calling (e.g., OpenAI, Gemini)
llm = ChatOpenAI(model=gpt_model, temperature=0)

# 2. Initialize the LangChain SQL Toolkit
# This toolkit will use the 'db' object with your custom schema info.
sql_toolkit = SQLDatabaseToolkit(db=db, llm=llm)

Setup the system prompt

In [15]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")
system_prompt = prompt_template.format(dialect="PostgreSQL", top_k=3)
system_prompt += "\nIf you receive an empty result from the SQL query, that is acceptable."

Specify the return data format

In [25]:
from pydantic import BaseModel, Field


class Room(BaseModel):
    number: int = Field(..., description="the room number")
    type: str = Field(..., description="the type of room")
    max_occupancy: int = Field(..., description="the maximum number of people allowed to stay in the room")
    bed_type: str = Field(..., description="the types of beds in the room")
    accessibility: bool = Field(description="whether the room is handicapped accessible")
    view_type: list[str] = Field(description="the type of view out the window of the room")
    date: str = Field(description="the date the listing is for")
    price: float = Field(description="the price to book the room for the day")

class Rooms(BaseModel):
    rooms: list[Room] = Field(..., description="details of the rooms")

Setup the LangChain agent

In [30]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=sql_toolkit.get_tools(),
    system_prompt=system_prompt,
    response_format=Rooms,
)

In [28]:
def stream_sql_agent_call(prompt):
    for step in agent.stream(
        {"messages": [{"role": "user", "content": prompt}]},
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()

In [31]:
prompt="How much to book the presidential suite for the week of January 5th?"
stream_sql_agent_call(prompt)

================================ Human Message =================================

How much to book the presidential suite for the week of January 5th?


BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'sql_db_list_tables': In context=(), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'tools[2].function.parameters', 'code': 'invalid_function_parameters'}}

In [29]:
prompt="How much to book the penthouse suite for the week of January 5th?"
stream_sql_agent_call(prompt)

================================ Human Message =================================

How much to book the penthouse suite for the week of January 5th?


BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'sql_db_list_tables': In context=(), 'additionalProperties' is required to be supplied and to be false.", 'type': 'invalid_request_error', 'param': 'tools[2].function.parameters', 'code': 'invalid_function_parameters'}}